# CKD Data Preprocessing

This notebook preprocesses the Chronic Kidney Disease (CKD) dataset before model training.

## Objectives

- Load the raw CKD dataset
- Handle missing values
- Encode categorical variables
- Convert all features into numerical format
- Save the processed dataset for machine learning experiments

Output:

data/processed/chronic_kidney_disease_processed.csv

In [16]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [17]:
df = pd.read_csv("../../data/raw/chronic_kidney_disease.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (400, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     391 non-null    float64
 1   bp      388 non-null    float64
 2   sg      353 non-null    float64
 3   al      354 non-null    float64
 4   su      351 non-null    float64
 5   rbc     248 non-null    str    
 6   pc      335 non-null    str    
 7   pcc     396 non-null    str    
 8   ba      396 non-null    str    
 9   bgr     356 non-null    float64
 10  bu      381 non-null    float64
 11  sc      383 non-null    float64
 12  sod     313 non-null    float64
 13  pot     312 non-null    float64
 14  hemo    348 non-null    float64
 15  pcv     329 non-null    float64
 16  wbcc    294 non-null    float64
 17  rbcc    269 non-null    float64
 18  htn     398 non-null    str    
 19  dm      398 non-null    str    
 20  cad     398 non-null    str    
 21  appet   399 non-null    str    
 22  pe      399 n

In [19]:
print("Missing values before replacing '?'")
print(df.isnull().sum())

print("\nDataset shape:", df.shape)

Missing values before replacing '?'
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64

Dataset shape: (400, 25)


In [20]:
# Replace '?' with missing values

df.replace("?", np.nan, inplace=True)

# Remove leading/trailing spaces

df = df.apply(
    lambda col: col.str.strip()
    if col.dtype == object
    else col
)

In [21]:
print("Missing values after replacing '?'")
print(df.isnull().sum())

Missing values after replacing '?'
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64


In [22]:
# Separate numerical and categorical columns

numerical_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_columns = df.select_dtypes(
    include=["object"]
).columns

print("Numerical Columns:")
print(numerical_columns)

print()

print("Categorical Columns:")
print(categorical_columns)

Numerical Columns:
Index(['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo',
       'pcv', 'wbcc', 'rbcc'],
      dtype='str')

Categorical Columns:
Index(['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane',
       'class'],
      dtype='str')


In [23]:
# Columns that should be numeric

numeric_columns = [
    "age","bp","sg","al","su","bgr","bu","sc",
    "sod","pot","hemo","pcv","wbcc","rbcc"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Remaining categorical columns

categorical_columns = [
    col for col in df.columns
    if col not in numeric_columns
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [24]:
from sklearn.preprocessing import LabelEncoder

target = "class"

feature_categorical = [
    col for col in categorical_columns
    if col != target
]

# Encode feature columns
for col in feature_categorical:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Encode target separately
target_encoder = LabelEncoder()
df[target] = target_encoder.fit_transform(df[target].astype(str))

print("Categorical encoding completed.")

Categorical encoding completed.


In [25]:
print(df.isnull().sum())

age      0
bp       0
sg       0
al       0
su       0
rbc      0
pc       0
pcc      0
ba       0
bgr      0
bu       0
sc       0
sod      0
pot      0
hemo     0
pcv      0
wbcc     0
rbcc     0
htn      0
dm       0
cad      0
appet    0
pe       0
ane      0
class    0
dtype: int64


In [26]:
print(df.head())

    age    bp     sg   al   su  rbc  pc  pcc  ba    bgr    bu   sc    sod  \
0  48.0  80.0  1.020  1.0  0.0    1   1    0   0  121.0  36.0  1.2  138.0   
1   7.0  50.0  1.020  4.0  0.0    1   1    0   0  121.0  18.0  0.8  138.0   
2  62.0  80.0  1.010  2.0  3.0    1   1    0   0  423.0  53.0  1.8  138.0   
3  48.0  70.0  1.005  4.0  0.0    1   0    1   0  117.0  56.0  3.8  111.0   
4  51.0  80.0  1.010  2.0  0.0    1   1    0   0  106.0  26.0  1.4  138.0   

   pot  hemo   pcv    wbcc  rbcc  htn  dm  cad  appet  pe  ane  class  
0  4.4  15.4  44.0  7800.0   5.2    1   1    0      0   0    0      0  
1  4.4  11.3  38.0  6000.0   4.8    0   0    0      0   0    0      0  
2  4.4   9.6  31.0  7500.0   4.8    0   1    0      1   0    1      0  
3  2.5  11.2  32.0  6700.0   3.9    1   0    0      1   1    1      0  
4  4.4  11.6  35.0  7300.0   4.6    0   0    0      0   0    0      0  


In [27]:
print("\nFinal dataset information")
print(df.info())

print("\nRemaining missing values:")
print(df.isnull().sum().sum())

print("\nDataset shape:")
print(df.shape)


Final dataset information
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     400 non-null    float64
 1   bp      400 non-null    float64
 2   sg      400 non-null    float64
 3   al      400 non-null    float64
 4   su      400 non-null    float64
 5   rbc     400 non-null    int64  
 6   pc      400 non-null    int64  
 7   pcc     400 non-null    int64  
 8   ba      400 non-null    int64  
 9   bgr     400 non-null    float64
 10  bu      400 non-null    float64
 11  sc      400 non-null    float64
 12  sod     400 non-null    float64
 13  pot     400 non-null    float64
 14  hemo    400 non-null    float64
 15  pcv     400 non-null    float64
 16  wbcc    400 non-null    float64
 17  rbcc    400 non-null    float64
 18  htn     400 non-null    int64  
 19  dm      400 non-null    int64  
 20  cad     400 non-null    int64  
 21  appet   400 non-null   

In [28]:
os.makedirs("../../data/processed", exist_ok=True)

df.to_csv(
    "../../data/processed/chronic_kidney_disease_processed.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [29]:
processed = pd.read_csv(
    "../../data/processed/chronic_kidney_disease_processed.csv"
)

print(processed.shape)

display(processed.head())

(400, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,1,1,0,0,121.0,36.0,1.2,138.0,4.4,15.4,44.0,7800.0,5.2,1,1,0,0,0,0,0
1,7.0,50.0,1.020,4.0,0.0,1,1,0,0,121.0,18.0,0.8,138.0,4.4,11.3,38.0,6000.0,4.8,0,0,0,0,0,0,0
2,62.0,80.0,1.010,2.0,3.0,1,1,0,0,423.0,53.0,1.8,138.0,4.4,9.6,31.0,7500.0,4.8,0,1,0,1,0,1,0
3,48.0,70.0,1.005,4.0,0.0,1,0,1,0,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,1,0,0,1,1,1,0
4,51.0,80.0,1.010,2.0,0.0,1,1,0,0,106.0,26.0,1.4,138.0,4.4,11.6,35.0,7300.0,4.6,0,0,0,0,0,0,0


In [30]:
print("=" * 50)
print("Preprocessing completed successfully")
print("=" * 50)

print("\nFinal Dataset Shape :", processed.shape)

print("\nMissing Values :", processed.isnull().sum().sum())

print("\nData Types")
print(processed.dtypes)

print("\nTarget Distribution")
print(processed["class"].value_counts())

Preprocessing completed successfully

Final Dataset Shape : (400, 25)

Missing Values : 0

Data Types
age      float64
bp       float64
sg       float64
al       float64
su       float64
rbc        int64
pc         int64
pcc        int64
ba         int64
bgr      float64
bu       float64
sc       float64
sod      float64
pot      float64
hemo     float64
pcv      float64
wbcc     float64
rbcc     float64
htn        int64
dm         int64
cad        int64
appet      int64
pe         int64
ane        int64
class      int64
dtype: object

Target Distribution
class
0    250
1    150
Name: count, dtype: int64
